<a href="https://colab.research.google.com/github/gadekarvishal08-cloud/Gen-AI-IN226005202/blob/main/gen_ai_task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langchain langchain-openai langsmith python-dotenv

In [ ]:
import os
from google.colab import userdata

# Load API Keys from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY')

# Enable LangSmith Tracing (Mandatory for your assignment)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Resume_Screening_Project" # Give it a custom tag
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

print("✅ Environment Variables Set & Tracing Enabled!")

In [ ]:
!mkdir -p prompts
!mkdir -p chains
!mkdir -p data

In [ ]:
%%writefile prompts/screening_prompt.py
from langchain_core.prompts import PromptTemplate

# We use JSON formatting instructions to get structured output
screening_template = """
You are an expert technical recruiter and Data Science hiring manager.
Your task is to evaluate a candidate's resume against a specific job description.

CRITICAL RULE: Do NOT assume skills, experience, or tools that are not explicitly present in the resume. No hallucinations.

Job Description:
{job_description}

Candidate Resume:
{resume}

Analyze the resume and output a JSON object containing the following keys:
1. "extracted_skills": A list of technical skills, tools, and experience extracted directly from the resume.
2. "matching_skills": A list of skills from the resume that match the Job Description.
3. "missing_skills": A list of key skills required by the Job Description that are missing from the resume.
4. "fit_score": An integer between 0 and 100 representing how well the candidate fits the role.
5. "explanation": A detailed, step-by-step reasoning for the assigned fit score.

Output exactly a JSON object and nothing else.
"""

prompt_template = PromptTemplate(
    input_variables=["job_description", "resume"],
    template=screening_template
)

In [ ]:
%%writefile chains/screening_chain.py
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from prompts.screening_prompt import prompt_template

def build_screening_chain():
    # Initialize the LLM (Using GPT-4o-mini for cost efficiency and good JSON adherence)
    # We use model_kwargs to enforce JSON output (Bonus feature)
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.1,
        model_kwargs={"response_format": {"type": "json_object"}}
    )

    # Initialize JSON Parser
    json_parser = JsonOutputParser()

    # Build the LCEL Pipeline: Prompt -> LLM -> JSON Parser
    chain = prompt_template | llm | json_parser

    return chain

In [ ]:
%%writefile main.py
import json
from chains.screening_chain import build_screening_chain
from langsmith import traceable

# 1. Define Mock Data
job_description = """
Data Scientist Role.
Requirements: 3+ years experience, Python, SQL, Machine Learning (Scikit-Learn, XGBoost),
Deep Learning (PyTorch or TensorFlow), LangChain, and API deployment.
Must have experience building LLM pipelines.
"""

resumes = {
    "Strong Candidate": """
    Senior Data Scientist with 4 years of experience.
    Skills: Python, SQL, PostgreSQL, Scikit-Learn, TensorFlow, PyTorch.
    Experience: Built production LLM pipelines using LangChain and OpenAI APIs. Deployed models via FastAPI.
    """,
    "Average Candidate": """
    Data Analyst / Junior Data Scientist with 2 years of experience.
    Skills: Python, SQL, Pandas, Tableau, Scikit-Learn.
    Experience: Built predictive churn models using XGBoost. Familiar with basic NLP but no hands-on LangChain experience.
    """,
    "Weak Candidate": """
    Recent Graduate.
    Skills: Java, C++, HTML, CSS, basic Python.
    Experience: Built a front-end website. Took one introductory course on Artificial Intelligence.
    """
}

# Wrap the main execution in a LangSmith traceable decorator
@traceable(name="Evaluate_All_Candidates")
def evaluate_candidates():
    print("🚀 Initializing AI Resume Screening Pipeline...\n")
    chain = build_screening_chain()

    results = {}

    for candidate_type, resume_text in resumes.items():
        print(f"📊 Evaluating: {candidate_type}...")

        # Invoke the LCEL chain
        response = chain.invoke({
            "job_description": job_description,
            "resume": resume_text
        })

        results[candidate_type] = response

        print(f"   Score: {response.get('fit_score')}/100")
        print(f"   Explanation: {response.get('explanation')}\n")

    return results

if __name__ == "__main__":
    final_results = evaluate_candidates()

    # Save results to a file (optional, but good for reporting)
    with open("data/screening_results.json", "w") as f:
        json.dump(final_results, f, indent=4)

    print("✅ All candidates processed. Traces sent to LangSmith!")

In [ ]:
!python main.py